# Observe, Predict, Act
**EXERCISE NOTEBOOK · ArtificialGate · Dr. Barak Or**

**Foundation of Physical AI · Physical AI Essentials, Lesson 1**

**Estimated participant work: 60 minutes.** Complete the marked TODO lines. Run supplied cells unchanged unless a task names a parameter to vary.
Times include reading, coding, reruns, and interpretation; download speed can vary.

**Skills you will practise:** Identify model roles; train an action-conditioned predictor; measure feedback and observation-delay effects.

**Before you start:** Read the matching book lesson; know NumPy arrays and PyTorch tensors/gradients.

## Your 60-Minute Task Plan
| Task | Estimated time |
|---|---:|
| PAI1-T01 · Set Up The Runtime | 4 min |
| PAI1-T02 · Identify Model Roles | 3 min |
| PAI1-T03 · Inspect Recorded Interactions | 4 min |
| PAI1-T04 · Build Valid Transitions | 7 min |
| PAI1-T05 · Complete The Training Step | 6 min |
| PAI1-T06 · Compare Learning Rates | 8 min |
| PAI1-T07 · Measure Prediction Drift | 6 min |
| PAI1-T08 · Complete The Feedback Controller | 5 min |
| PAI1-T09 · Test Feedback And Observation Delay | 12 min |
| PAI1-T10 · Save And Explain | 5 min |
| **Total** | **60 min** |

### Task PAI1-T01 — Set Up The Runtime · 4 min
**Do:** Run setup; keep the seed fixed.  
**Produce:** A ready message with package versions.  
**Check:** No import error; existing core packages are preserved.

In [ ]:
#@title Setup — run once (expand to inspect dependencies)
import os, sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path
if os.environ.get("AG_LOCAL_TEST") != "1":
    # Preserve Colab's existing NumPy/Pandas/PyTorch ABI. Install only missing modules.
    required = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
                "pyarrow": "pyarrow>=18,<24", "huggingface_hub": "huggingface_hub>=0.30,<2",
                "torch": "torch>=2.5,<3", "gymnasium": "gymnasium>=0.29,<1.3",
                "cv2": "opencv-python-headless==4.11.0.86", "pygame": "pygame==2.6.1",
                "shapely": "shapely>=2,<3", "cffi": "cffi", "IPython": "ipython"}
    missing = [package for module, package in required.items()
               if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    # Legacy PushT contact APIs require Pymunk 6. Keep compatibility pins private.
    critical = {"gym-pusht": "0.1.8", "pymunk": "6.11.1"}
    def installed_version(package):
        try: return importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: return None
    if any(installed_version(k) != v for k, v in critical.items()):
        if any(name in sys.modules for name in ["pymunk", "gym_pusht"]):
            raise RuntimeError("Restart this notebook runtime once; an incompatible physics module was already imported.")
        overlay = Path.cwd() / ".artificialgate_lab1_deps_v3"
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                               "--target", str(overlay), "--upgrade",
                               *[f"{k}=={v}" for k,v in critical.items()]])
        sys.path.insert(0, str(overlay)); importlib.invalidate_caches()
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
import copy, hashlib, json, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import display, HTML
import torch
from torch import nn
from huggingface_hub import hf_hub_download
from importlib.metadata import version
for package, expected in {"gym-pusht": "0.1.8", "pymunk": "6.11.1"}.items():
    assert version(package) == expected, f"{package} must be {expected}; rerun Setup and restart the runtime if already imported."
torch.set_num_threads(2)
DEVICE = torch.device("cpu")   # This small model is deliberately CPU-first.
SEED = 27
def seed_all(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
seed_all()
BLUE, NAVY, CYAN, GREY = "#1647FF", "#071738", "#00B9E8", "#8A96AA"
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=[BLUE, CYAN, NAVY, GREY]),
                     "axes.spines.top": False, "axes.spines.right": False,
                     "figure.dpi": 115, "font.size": 10})
OUT = Path("lab1_results"); OUT.mkdir(exist_ok=True)
print(f"PyTorch {torch.__version__} | {DEVICE} | seed {SEED}")

### Task PAI1-T02 — Identify Model Roles · 3 min
**Do:** Run the checker; complete the five role cases using the supplied vocabulary. A component can have overlapping tags.  
**Produce:** A checked model-role table.  
**Check:** Distinguish learned prediction, action selection, simulation, and asset-specific correspondence.

In [ ]:
#@title Role-check feedback — supplied checker (tags can overlap)
ROLE_OPTIONS = {"world_model", "VLA", "policy", "simulator", "JEPA", "foundation_model", "digital_twin"}
def check_component_roles(answers):
    expected = [{"world_model"}, {"VLA", "policy"}, {"simulator"},
                {"JEPA", "world_model", "foundation_model"}, {"simulator", "digital_twin"}]
    assert len(answers) == len(expected)
    for index, (answer, target) in enumerate(zip(answers, expected), 1):
        assert isinstance(answer, set), f"Case {index}: supply a set of role tags."
        assert answer <= ROLE_OPTIONS, f"Case {index}: use the supplied tag vocabulary."
        assert answer == target, f"Case {index}: revisit its inputs, outputs and transfer/asset scope."
    return {"passed": True, "cases": len(expected), "tags": [sorted(a) for a in answers]}

In [ ]:
# Role check — 2 min. Select SETS from ROLE_OPTIONS; categories need not be exclusive.
print("Available tags:", sorted(ROLE_OPTIONS))
component_roles = [
    None,  # observation/action history -> next object-state prediction
    None,  # image + instruction -> robot action distribution
    None,  # encoded contact physics + action -> next simulated state
    None,  # JEPA context/target encoders; broadly reusable action-conditioned future embeddings
    None,  # simulator kept synchronized with one identified physical asset
]
role_evidence = check_component_roles(component_roles)
display(pd.DataFrame({"component": ["Our dynamics network", "Vision-language action policy",
                                    "Contact physics engine", "Broad predictive embedding model",
                                    "Asset-synchronized simulator"],
                      "roles": [", ".join(sorted(r)) for r in component_roles]}))

### Task PAI1-T03 — Inspect Recorded Interactions · 4 min
**Do:** Load one episode: two-frame observation history; absolute target actions; simulation units, not meters.  
**Produce:** An episode plot and split counts.  
**Check:** Each observation has 18 values; each action has 2.

In [ ]:
#@title Load the pinned public dataset and build episode-level splits
DATASET = "lerobot/pusht_keypoints"
REVISION = "ace8c161a68bc025c21a5f29f85b86a9a2c5e64b"
data_file = hf_hub_download(DATASET, "data/chunk-000/file-000.parquet",
                            repo_type="dataset", revision=REVISION)
df = pd.read_parquet(data_file).sort_values(["episode_index", "frame_index"]).reset_index(drop=True)
required = {"observation.state", "observation.environment_state", "action",
            "episode_index", "frame_index", "timestamp"}
assert required <= set(df), "Dataset schema changed. Do not silently substitute columns."
states = np.concatenate([np.stack(df["observation.state"]),
                         np.stack(df["observation.environment_state"])], axis=1).astype("float32")
actions = np.stack(df["action"]).astype("float32")
episode = df.episode_index.to_numpy(); frame = df.frame_index.to_numpy()
assert states.shape[1] == 18 and actions.shape[1] == 2
assert np.isfinite(states).all() and np.isfinite(actions).all()
ids = np.random.default_rng(SEED).permutation(np.unique(episode))
n_train, n_val = int(.70 * len(ids)), int(.15 * len(ids))
split_ids = {"train": ids[:n_train], "validation": ids[n_train:n_train+n_val],
             "test": ids[n_train+n_val:]}
assert not (set(split_ids["train"]) & set(split_ids["test"]))
file_sha256 = hashlib.sha256(Path(data_file).read_bytes()).hexdigest()
assert file_sha256 == "8b4d978eeeb99254ddead01e79ee70d95a0416e27438ca9438c89a1682598ae0", "Dataset integrity check failed."
provenance = {"dataset": DATASET, "revision": REVISION,
              "file_sha256": file_sha256,
              "license": "Apache-2.0", "source": "public simulated PushT demonstrations",
              "splits": {k: v.tolist() for k, v in split_ids.items()}}
(OUT / "data_manifest.json").write_text(json.dumps(provenance, indent=2))
display(pd.DataFrame({"split": list(split_ids), "episodes": [len(v) for v in split_ids.values()]}))
selected_episode = np.flatnonzero(episode == split_ids["train"][0])
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(*states[selected_episode, :2].T, label="Agent")
centers = states[selected_episode, 2:].reshape(-1, 8, 2).mean(1)
ax.plot(*centers.T, label="Object keypoint centre")
ax.set(xlabel="x [simulation units]", ylabel="y [simulation units]", title="One recorded PushT episode")
ax.set_aspect("equal"); ax.invert_yaxis(); ax.legend()
fig.savefig(OUT / "dataset_episode.png", bbox_inches="tight"); plt.show()

### Task PAI1-T04 — Build Valid Transitions · 7 min
**Do:** Complete TODO 1a/1b: keep within-episode consecutive samples and calculate next-state minus current-state.  
**Produce:** Inputs and state-change targets.  
**Check:** Transition tests pass without crossing resets.

In [ ]:
# Task 1 — build valid transitions; never cross an episode reset.
same_episode = (episode[:-2] == episode[1:-1]) & (episode[1:-1] == episode[2:])
consecutive = (np.diff(frame)[:-1] == 1) & (np.diff(frame)[1:] == 1)
valid = None  # TODO 1a: combine the two Boolean masks
assert valid is not None, "Complete TODO 1a, then rerun this cell."
assert np.array_equal(valid, same_episode & consecutive), "History and target must be consecutive in one episode."
previous_state, current_state, next_state = states[:-2][valid], states[1:-1][valid], states[2:][valid]
state_change = None  # TODO 1b: the observed state change
assert state_change is not None, "Complete TODO 1b, then rerun this cell."
assert np.allclose(state_change, next_state-current_state), "Target = next state minus current state."
X = np.concatenate([previous_state, current_state, actions[1:-1][valid]], axis=1) / 512.0
Y = state_change / 512.0
pair_episode = episode[1:-1][valid]
masks = {k: np.isin(pair_episode, v) for k, v in split_ids.items()}
x_mean = X[masks["train"]].mean(0); x_std = X[masks["train"]].std(0).clip(.01)
y_mean = Y[masks["train"]].mean(0); y_std = Y[masks["train"]].std(0).clip(.002)
tensors = {k: (torch.tensor((X[m]-x_mean)/x_std), torch.tensor((Y[m]-y_mean)/y_std))
           for k, m in masks.items()}
assert sum(m.sum() for m in masks.values()) == len(X)
print(f"Valid transitions: {len(X):,} | inputs {X.shape[1]} | outputs {Y.shape[1]}")

### Task PAI1-T05 — Complete The Training Step · 6 min
**Do:** Complete MSE, backpropagation, and optimizer update; run the supplied helper.  
**Produce:** A working learning step.  
**Check:** Real model weights change.

In [ ]:
class DynamicsModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(38, 128), nn.ReLU(),
                                 nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 18))
    def forward(self, x):
        return self.net(x)

def training_step(model, optimizer, inputs, target):
    optimizer.zero_grad()
    prediction = model(inputs)
    loss = None  # TODO 2a: mean squared prediction error
    assert loss is not None, "Complete TODO 2a."
    raise NotImplementedError('TODO 2b: compute loss gradients')
    raise NotImplementedError('TODO 2c: update model parameters')
    return float(loss.detach())

probe = DynamicsModel(); before = copy.deepcopy(probe.state_dict())
probe_optimizer = torch.optim.Adam(probe.parameters(), lr=.001)
training_step(probe, probe_optimizer, *[v[:8] for v in tensors["train"]])
assert any(not torch.equal(before[k], probe.state_dict()[k]) for k in before), "No trainable weight changed."
print("Training-step test passed: real parameters updated.")

In [ ]:
#@title Training runner — model/optimizer/checkpoint logic is provided
def train_dynamics(learning_rate, steps=400, batch_size=256):
    seed_all()
    model = DynamicsModel().to(DEVICE)
    initial_state = copy.deepcopy(model.state_dict())
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    tx, ty = tensors["train"]; vx, vy = tensors["validation"]
    gen = torch.Generator().manual_seed(SEED)
    history, best_loss, best_state = [], float("inf"), None
    started = time.perf_counter()
    for step in range(steps+1):
        if step % 25 == 0:
            model.eval()
            with torch.no_grad():
                train_loss = nn.functional.mse_loss(model(tx[:2048]), ty[:2048]).item()
                val_loss = nn.functional.mse_loss(model(vx), vy).item()
            history.append({"step": step, "train_loss": train_loss, "validation_loss": val_loss})
            if val_loss < best_loss:
                best_loss, best_state = val_loss, copy.deepcopy(model.state_dict())
        if step == steps:
            break
        idx = torch.randint(len(tx), (batch_size,), generator=gen)
        model.train(); training_step(model, optimizer, tx[idx], ty[idx])
    model.load_state_dict(best_state); model.eval()
    weights_changed = any(not torch.equal(initial_state[k], model.state_dict()[k]) for k in initial_state)
    assert weights_changed, "Training did not change the selected checkpoint."
    return {"model": model, "history": pd.DataFrame(history), "lr": learning_rate,
            "validation_loss": best_loss, "weights_changed": weights_changed,
            "seconds": time.perf_counter()-started}

@torch.no_grad()
def predict_next(model, previous, current, action):
    previous = np.atleast_2d(previous).astype("float32")
    current = np.atleast_2d(current).astype("float32")
    action = np.atleast_2d(action).astype("float32")
    batch = np.concatenate([previous, current, action], axis=1) / 512.0
    prediction = model(torch.tensor((batch-x_mean)/x_std)).numpy()
    return current + (prediction*y_std+y_mean)*512.0

### Task PAI1-T06 — Compare Learning Rates · 8 min
**Do:** Run both rates from identical initial weights and a fixed training budget.  
**Produce:** Training/validation curves.  
**Check:** Choose by validation loss, not test results.

In [ ]:
# Experiment A — change only LEARNING_RATES; use the same seed and training budget.
LEARNING_RATES = [3e-4, 3e-3]
TRAINING_STEPS = 400
runs = [train_dynamics(lr, steps=TRAINING_STEPS) for lr in LEARNING_RATES]
best = min(runs, key=lambda r: r["validation_loss"])
model = best["model"]
fig, ax = plt.subplots(figsize=(7, 3.5))
for result in runs:
    h = result["history"]
    ax.plot(h.step, h.validation_loss, label=f"Validation · lr={result['lr']:g}")
    ax.plot(h.step, h.train_loss, "--", alpha=.65, label=f"Train · lr={result['lr']:g}")
ax.set(xlabel="Optimizer steps", ylabel="Standardized delta MSE", title="Actual neural-network training")
ax.legend(fontsize=8)
fig.savefig(OUT / "training_curves.png", bbox_inches="tight"); plt.show()
display(pd.DataFrame([{"learning_rate": f"{r['lr']:.0e}",
                       "validation_loss": r["validation_loss"], "seconds": r["seconds"]}
                      for r in runs]).round(3))

### Task PAI1-T07 — Measure Prediction Drift · 6 min
**Do:** Compare held-out predictions with persistence across horizons; run the supplied environment helper.  
**Produce:** Error table and horizon graph.  
**Check:** Prediction error and task success remain separate metrics.

In [ ]:
#@title Held-out prediction and free-running rollout evaluation
chosen = masks["test"]
test_prev, test_now, test_next = previous_state[chosen], current_state[chosen], next_state[chosen]
test_action = actions[1:-1][valid][chosen]
neural_next = predict_next(model, test_prev, test_now, test_action)
def object_rmse(prediction, truth):
    return np.sqrt(np.mean((prediction[:, 2:]-truth[:, 2:])**2))
offline = pd.DataFrame([
    {"predictor": "Persistence", "object coordinate RMSE": object_rmse(test_now, test_next)},
    {"predictor": "Neural dynamics", "object coordinate RMSE": object_rmse(neural_next, test_next)}])
display(offline.round(3))
horizon_rows = []
for horizon in [1, 3, 6, 12]:
    errors, persistence_errors = [], []
    for eid in split_ids["test"][:12]:
        ix = np.flatnonzero(episode == eid)
        if len(ix) < horizon+3: continue
        # Recorded future actions are the known input sequence, never future observations.
        for start in range(1, min(len(ix)-horizon, 45), 8):
            prev, now = states[ix[start-1]].copy(), states[ix[start]].copy()
            original = now.copy()
            for offset in range(horizon):
                nxt = predict_next(model, prev, now, actions[ix[start+offset]])[0]
                prev, now = now, nxt
            truth = states[ix[start+horizon]]
            errors.append(np.sqrt(np.mean((now[2:]-truth[2:])**2)))
            persistence_errors.append(np.sqrt(np.mean((original[2:]-truth[2:])**2)))
    horizon_rows.append({"horizon": horizon, "neural_RMSE": np.mean(errors),
                         "persistence_RMSE": np.mean(persistence_errors)})
horizon_table = pd.DataFrame(horizon_rows)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(horizon_table.horizon, horizon_table.neural_RMSE, "o-", label="Learned dynamics")
ax.plot(horizon_table.horizon, horizon_table.persistence_RMSE, "o--", label="Persistence")
ax.set(xlabel="Prediction horizon [control steps]", ylabel="Object-coordinate RMSE [simulation units]",
       title="Prediction error accumulates without fresh observations")
ax.legend()
fig.savefig(OUT / "prediction_horizons.png", bbox_inches="tight"); plt.show()

In [ ]:
#@title Provided physical environment and sampled-action planner
import gymnasium as gym
import gym_pusht

def pack_observation(obs):
    return np.concatenate([obs["agent_pos"], obs["environment_state"]]).astype("float32")

def make_local_task(seed):
    env = gym.make("gym_pusht/PushT-v0", obs_type="environment_state_agent_pos",
                   render_mode="rgb_array", visualization_width=256, visualization_height=256)
    rng = np.random.default_rng(seed)
    start_state = [130, 280, 256+rng.uniform(-12,12), 250+rng.uniform(-10,10), rng.uniform(-.12,.12)]
    obs, info = env.reset(seed=seed, options={"reset_to_state": start_state})
    # Initialize the agent beside the observed left bar, not inside the object.
    points = np.asarray(obs["environment_state"]).reshape(8, 2)
    target_y = points[np.argsort(points[:, 0])[:2], 1].mean()
    env.unwrapped.agent.position = (float(points[:, 0].min()-23), float(target_y))
    env.unwrapped.agent.velocity = (0, 0)
    obs = env.unwrapped.get_obs()
    state = pack_observation(obs)
    goal = state[2:].reshape(8, 2).mean(0) + np.array([30, 0])
    return env, state, goal

def candidate_plans(current, horizon, seed):
    rng = np.random.default_rng(seed)
    # Short nearby targets, including forward, sideways and hold candidates.
    direction = np.array([[0, 0], [18, 0], [30, 0], [42, 0], [24, -12], [24, 12]], dtype="float32")
    directions = np.concatenate([direction, rng.uniform([-5, -15], [45, 15], (58, 2))])
    plans = current[:2][None, None, :] + np.arange(1, horizon+1)[None, :, None]*directions[:, None, :]
    return np.clip(plans, 8, 504).astype("float32")

def plan_actions(model, previous, current, goal, horizon, seed):
    plans = candidate_plans(current, horizon, seed)
    prev = np.repeat(previous[None], len(plans), axis=0)
    now = np.repeat(current[None], len(plans), axis=0)
    for t in range(horizon):
        nxt = predict_next(model, prev, now, plans[:, t])
        prev, now = now, nxt
    centre = now[:, 2:].reshape(-1, 8, 2).mean(1)
    terminal_error = np.sum((centre-goal)**2, axis=1)
    # Avoid extrapolated geometry distortions masquerading as useful motion.
    shape_before = current[2:].reshape(8, 2).copy(); shape_before -= shape_before.mean(0)
    shape_after = now[:, 2:].reshape(-1, 8, 2)-centre[:, None]
    geometry_penalty = np.mean((shape_after-shape_before)**2, axis=(1, 2))
    effort = np.mean(np.sum(np.diff(np.concatenate([np.repeat(current[:2][None,None], len(plans), 0), plans], 1), axis=1)**2, axis=2), axis=1)
    score = terminal_error + 2*geometry_penalty + .002*effort
    assert np.isfinite(score).all(), "Invalid learned rollout; inspect the training curves."
    return plans[int(np.argmin(score))]

### Task PAI1-T08 — Complete The Feedback Controller · 5 min
**Do:** Complete first-action selection; run the queued-observation helper.  
**Produce:** A valid executable target.  
**Check:** Planning never reads future simulator states.

In [ ]:
# Task 3 — a receding-horizon controller executes one action, then observes again.
def first_action(planned):
    action = None  # TODO 3: execute only the first planned action
    assert action is not None, "Complete TODO 3: select the first action, not the full sequence."
    assert np.asarray(action).shape == (2,)
    return action
assert np.array_equal(first_action(np.array([[10, 20], [30, 40]])), [10, 20])
print("Action-interface test passed.")

In [ ]:
#@title Matched-condition physical evaluation and animation helpers
def delayed_observation_pair(observation_log, delay_steps):
    assert isinstance(delay_steps, int) and delay_steps >= 0
    current_index = max(0, len(observation_log)-1-delay_steps)
    previous_index = max(0, current_index-1)
    return (observation_log[previous_index].copy(), observation_log[current_index].copy(),
            previous_index, current_index)

queue_probe = [np.full(18, k, dtype="float32") for k in range(5)]
probe_previous, probe_current, probe_pi, probe_ci = delayed_observation_pair(queue_probe, 3)
assert probe_pi == 0 and probe_ci == 1
assert np.all(probe_previous == 0) and np.all(probe_current == 1)

def run_control(model, horizon, feedback, seed=101, disturb=True, steps=36, record=False,
                observation_delay=0):
    env, now, goal = make_local_task(seed)
    trace = [now.copy()]; predicted_trace = [now.copy()]
    observation_log = [now.copy()]
    used_history_steps, used_observation_steps = [], []
    frames, planning_calls, contacts, violations = [], 0, 0, 0
    plan, index = None, 0
    start = time.perf_counter()
    try:
        for step in range(steps):
            # Both history frames are queued. Fresh state is evaluation-only below.
            queued_previous, queued_current, queued_pi, queued_ci = delayed_observation_pair(
                observation_log, observation_delay)
            if feedback:
                prev_belief, belief = queued_previous, queued_current
                used_history_steps.append(queued_pi); used_observation_steps.append(queued_ci)
            else:
                belief = predicted_trace[-1]
                prev_belief = predicted_trace[-2] if len(predicted_trace)>1 else belief
                used_history_steps.append(-1); used_observation_steps.append(-1)
            if feedback or plan is None or index == len(plan):
                # Open-loop plans use the network's previous prediction rather than fresh state.
                plan = plan_actions(model, prev_belief.copy(), belief.copy(), goal, horizon, seed+step)
                index = 0; planning_calls += 1
            action = first_action(plan) if feedback else plan[index]
            index += 1
            predicted = predict_next(model, prev_belief, belief, action)[0]
            if disturb and step == 12:
                # Same exogenous disturbance in both conditions: no new training labels.
                env.unwrapped.agent.position += (0, 24)
            obs, reward, terminated, truncated, info = env.step(action)
            nxt = pack_observation(obs)
            contacts += int(info.get("n_contacts", 0) > 0)
            violations += int((action < 0).any() or (action > 512).any())
            if record: frames.append(env.render())
            now = nxt
            observation_log.append(now.copy())
            trace.append(now.copy()); predicted_trace.append(predicted.copy())
            if terminated or truncated: break
    finally:
        env.close()
    terminal_error = float(np.linalg.norm(now[2:].reshape(8,2).mean(0)-goal))
    return {"feedback": feedback, "horizon": horizon, "seed": seed, "disturbance": disturb,
            "observation_delay": observation_delay,
            "terminal_error": terminal_error, "local_success": terminal_error<=15,
            "contact_steps": contacts, "action_violations": violations,
            "planning_calls": planning_calls, "seconds": time.perf_counter()-start,
            "trace": np.array(trace), "predicted": np.array(predicted_trace), "goal": goal,
            "used_history_steps": used_history_steps, "used_observation_steps": used_observation_steps,
            "frames": frames}

def show_control(rows):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    all_points = np.concatenate([np.concatenate([
        row["trace"][:, 2:].reshape(-1,8,2).mean(1),
        row["predicted"][:, 2:].reshape(-1,8,2).mean(1),
        row["goal"][None]]) for row in rows])
    low, high = all_points.min(0)-5, all_points.max(0)+5
    for row, ax in zip(rows, axes):
        actual = row["trace"][:, 2:].reshape(-1,8,2).mean(1)
        predicted = row["predicted"][:, 2:].reshape(-1,8,2).mean(1)
        ax.plot(*actual.T, "o-", ms=2, label="Observed object centre")
        ax.plot(*predicted.T, "--", label="Predicted centre")
        ax.scatter(*row["goal"], marker="*", s=130, color=NAVY, label="Local goal")
        ax.set(title="Feedback" if row["feedback"] else "Open loop", xlabel="x [simulation units]", ylabel="y [simulation units]")
        ax.set_xlim(low[0], high[0]); ax.set_ylim(high[1], low[1])
        ax.set_aspect("equal"); ax.grid(alpha=.15)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=8)
    plt.tight_layout(rect=(0, .08, 1, 1))
    fig.savefig(OUT / "control_comparison.png", bbox_inches="tight"); plt.show()

def display_animation(frames):
    if not frames: return
    fig, ax = plt.subplots(figsize=(3.5,3.5)); ax.axis("off")
    artist = ax.imshow(frames[0])
    def update(i): artist.set_data(frames[i]); return (artist,)
    animation = FuncAnimation(fig, update, frames=len(frames), interval=100, blit=True)
    plt.close(fig); display(HTML(animation.to_jshtml()))

### Task PAI1-T09 — Test Feedback And Observation Delay · 12 min
**Do:** Run the horizon comparison, then delays of 0, 1, and 3 samples. Keep model, seeds, and disturbance fixed.  
**Produce:** Trajectory, animation, and error-versus-delay graphs.  
**Check:** Both history frames are delayed; report age in milliseconds. Local success is not full benchmark success.

In [ ]:
# Experiment B — change only HORIZONS. Feedback uses additional planning calls; report them.
HORIZONS = [2, 5]
EVALUATION_SEEDS = [101, 102, 103]
control_results = [run_control(model, horizon, feedback, seed, disturb=True)
                   for horizon in HORIZONS for seed in EVALUATION_SEEDS for feedback in [False, True]]
summary_columns = ["feedback", "horizon", "seed", "terminal_error", "local_success",
                   "contact_steps", "action_violations", "planning_calls", "seconds"]
control_table = pd.DataFrame([{k:r[k] for k in summary_columns} for r in control_results])
display(control_table.groupby(["feedback", "horizon"]).agg(
    terminal_error=("terminal_error", "mean"), local_success=("local_success", "mean"),
    contact_steps=("contact_steps", "mean"), planning_calls=("planning_calls", "mean")).round(3))
examples = [run_control(model, HORIZONS[0], feedback, 101, disturb=True, record=True)
            for feedback in [False, True]]
show_control(examples)
display_animation(examples[1]["frames"])

In [ ]:
# Experiment C — same trained model, task, seeds, horizon and disturbance; only delay changes.
OBSERVATION_DELAYS = [0, 1, 3]     # control steps; each step is 100 ms
DELAY_HORIZON = HORIZONS[0]
delay_results = [run_control(model, DELAY_HORIZON, True, seed, disturb=True,
                             observation_delay=int(delay))
                 for delay in OBSERVATION_DELAYS for seed in EVALUATION_SEEDS]
for result in delay_results:
    expected_current = [max(0, step-result["observation_delay"])
                        for step in range(len(result["used_observation_steps"]))]
    assert result["used_observation_steps"] == expected_current
    assert result["used_history_steps"] == [max(0, index-1) for index in expected_current]
delay_queue_checks_passed = True
delay_table = pd.DataFrame([{k: r[k] for k in ["observation_delay", "horizon", "seed",
    "disturbance", "terminal_error", "local_success", "contact_steps", "planning_calls"]}
    for r in delay_results])
delay_summary = delay_table.groupby("observation_delay").agg(
    terminal_error=("terminal_error", "mean"), local_success=("local_success", "mean"))
display(delay_summary.round(3))
fig, ax = plt.subplots(figsize=(6, 3.4))
for seed, group in delay_table.groupby("seed"):
    ax.plot(group.observation_delay*100, group.terminal_error, "o-", alpha=.35,
            label=f"Reset {seed}")
ax.plot(delay_summary.index*100, delay_summary.terminal_error, "o-", lw=2.5,
        color=NAVY, label="Mean of matched resets")
ax.set(xlabel="Observation delay [ms]", ylabel="Terminal goal error [simulation units]",
       title="The same model acts on progressively older observations")
ax.legend(fontsize=8); ax.grid(alpha=.15)
fig.savefig(OUT / "observation_delay.png", bbox_inches="tight"); plt.show()

### Task PAI1-T10 — Save And Explain · 5 min
**Do:** Write three evidence-based sentences about rate, horizon, and feedback/delay; export.  
**Produce:** Checkpoint, results, role checks, and skills evidence.  
**Check:** Reloaded predictions match; conclusions cite measured results.

In [ ]:
ANSWERS = {"learning_rate": "", "horizon": "", "feedback": ""}
torch.save({"state_dict": model.state_dict(), "x_mean": torch.tensor(x_mean), "x_std": torch.tensor(x_std),
            "y_mean": torch.tensor(y_mean), "y_std": torch.tensor(y_std),
            "dataset_revision": REVISION, "learning_rate": best["lr"]}, OUT / "dynamics_checkpoint.pt")
reloaded = torch.load(OUT / "dynamics_checkpoint.pt", map_location="cpu", weights_only=True)
restored_model = DynamicsModel(); restored_model.load_state_dict(reloaded["state_dict"]); restored_model.eval()
with torch.no_grad():
    assert torch.equal(model(tensors["test"][0][:8]), restored_model(tensors["test"][0][:8])), "Reloaded model changed predictions."
assert np.allclose(reloaded["x_mean"].numpy(), x_mean) and np.allclose(reloaded["y_std"].numpy(), y_std)
pd.concat([r["history"].assign(learning_rate=r["lr"]) for r in runs]).to_csv(OUT / "training.csv", index=False)
horizon_table.to_csv(OUT / "prediction_horizons.csv", index=False)
control_table.to_csv(OUT / "control_results.csv", index=False)
delay_table.to_csv(OUT / "observation_delay.csv", index=False)
skill_evidence = {
    "book": "Physical AI Essentials", "lesson": 1, "role_classification": role_evidence,
    "runtime_versions": {"python": sys.version.split()[0], "torch": torch.__version__,
                         "numpy": np.__version__, "pandas": pd.__version__,
                         "gym-pusht": version("gym-pusht"), "pymunk": version("pymunk")},
    "training": {"trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
                 "selected_learning_rate": best["lr"],
                 "all_runs_changed_weights": all(r["weights_changed"] for r in runs),
                 "checkpoint_reload_predictions_identical": True},
    "prediction": {"one_step": offline.to_dict("records"), "horizons": horizon_table.to_dict("records")},
    "physical_evaluation": {"horizon_sweep": control_table.to_dict("records"),
                            "observation_delay": delay_table.to_dict("records"),
                            "queued_both_history_frames": delay_queue_checks_passed},
    "scope": "Local 30-unit pushing task, 3 matched resets; not full PushT benchmark certification"
}
(OUT / "skill_evidence.json").write_text(json.dumps(skill_evidence, indent=2))
(OUT / "answers.json").write_text(json.dumps(ANSWERS, indent=2))
print("Saved checkpoint, split manifest and measured results in", OUT.resolve())

**Sources:** [PushT dataset](https://huggingface.co/datasets/lerobot/pusht_keypoints)
· [Gym PushT](https://github.com/huggingface/gym-pusht)
· [Learning models for control](https://underactuated.mit.edu/sysid.html).
**Book:** Lesson 1, “From content generation to causal intervention”, “World model:
consequence prediction”, “The complete engineering stack”, and “Levels of success”.